In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import cv2, numpy as np
from sponsortrack import config
from sponsortrack.ingestion.logo_loader import load_logo

sift = cv2.SIFT_create()
bf = cv2.BFMatcher(cv2.NORM_L2)

def guardar_png(ruta, img):
    ok, buf = cv2.imencode(".png", img)
    buf.tofile(str(ruta))

def preparar_logo(archivo, solo_borde):
    img, mask = load_logo(config.LOGOS_DIR / archivo, solo_borde)
    w = config.SIFT_LOGO_WIDTH
    h = int(img.shape[0] * w / img.shape[1])
    g = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (w, h), interpolation=cv2.INTER_AREA)
    m = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
    kp, des = sift.detectAndCompute(g, m)
    return {"nombre": archivo, "kp": kp, "des": des, "size": (w, h)}

def mascara_frame(frame, excluir_arriba=True):
    m = np.full(frame.shape[:2], 255, np.uint8)
    if excluir_arriba:
        m[: int(frame.shape[0] * config.EXCLUDE_TOP_FRAC)] = 0
    return m

def evaluar_frame(frame, logos, excluir_arriba=True):
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    kf, df = sift.detectAndCompute(g, mascara_frame(frame, excluir_arriba))
    if df is None or len(kf) < 2:
        return None
    area_min = config.MIN_LOGO_AREA_PX * (frame.shape[1] / config.REF_WIDTH) ** 2
    mejor = None
    for lg in logos:
        if lg["des"] is None or len(lg["des"]) < 2:
            continue
        pares = bf.knnMatch(lg["des"], df, k=2)
        buenos = [p[0] for p in pares
                  if len(p) == 2 and p[0].distance < config.LOWE_RATIO * p[1].distance]
        if len(buenos) < config.MIN_GOOD_MATCHES:
            continue
        src = np.float32([lg["kp"][m.queryIdx].pt for m in buenos]).reshape(-1, 1, 2)
        dst = np.float32([kf[m.trainIdx].pt for m in buenos]).reshape(-1, 1, 2)
        H, inl = cv2.findHomography(src, dst, cv2.RANSAC, config.RANSAC_REPROJ_THRESH)
        if H is None:
            continue
        n = int(inl.sum())
        w, h = lg["size"]
        esq = cv2.perspectiveTransform(
            np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2), H)
        if n < config.MIN_INLIERS or not cv2.isContourConvex(esq) \
                or cv2.contourArea(esq) < area_min:
            continue
        if mejor is None or n > mejor["inliers"]:
            mejor = {"logo": lg["nombre"], "inliers": n, "esquinas": esq.reshape(-1, 2)}
    return mejor

In [2]:
fcb = config.BRANDS["fcb"]
logos = [preparar_logo(a, fcb["fondo_solo_borde"]) for a in fcb["variantes"]]

video = config.VIDEOS_DIR / "VideoTarget.mp4"
cap = cv2.VideoCapture(str(video))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
paso = config.TARGET_FPS * 3           # un fotograma cada 3 s
config.OUT_FIGURES.mkdir(parents=True, exist_ok=True)

resultados, muestras = [], 0
for i in range(total):
    if not cap.grab():
        break
    if i % paso:
        continue
    ok, frame = cap.retrieve()
    if not ok:
        continue
    muestras += 1
    r = evaluar_frame(frame, logos)
    if r:
        t = i / config.TARGET_FPS
        etiqueta = f"{int(t // 60)}m{int(t % 60):02d}s"
        r["t"] = etiqueta
        resultados.append(r)
        if len(resultados) <= 40:       # guarda hasta 40 imágenes con el contorno
            vis = frame.copy()
            cv2.polylines(vis, [r["esquinas"].astype(np.int32)], True, (0, 255, 0), 3)
            guardar_png(config.OUT_FIGURES / f"humo_{etiqueta}_{r['inliers']}inl.png", vis)
    if muestras % 50 == 0:
        print(f"{muestras} muestras, {len(resultados)} detecciones")

print(f"\n{len(resultados)} detecciones en {muestras} fotogramas muestreados")
for r in sorted(resultados, key=lambda r: -r["inliers"])[:10]:
    print(f"  {r['t']}  {r['logo']}  inliers={r['inliers']}")

50 muestras, 2 detecciones
100 muestras, 2 detecciones
150 muestras, 2 detecciones
200 muestras, 2 detecciones
250 muestras, 2 detecciones
300 muestras, 2 detecciones

2 detecciones en 345 fotogramas muestreados
  2m24s  fcb_clasico.png  inliers=40
  2m27s  fcb_clasico.png  inliers=38
